In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import sys, random, subprocess, importlib.metadata as metadata
from pathlib import Path
import numpy as np
import torch

SEED = 266
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
roots = [Path.cwd(), *Path.cwd().parents,
         Path("/content/data266-lab2-team41"),
         Path("/kaggle/working/data266-lab2-team41")]
ROOT = next((p for p in roots if (p / "src/p2_partner_baseline.py").is_file()), None)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip() if ROOT else "unavailable"
print("Git commit:", commit)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("SEED:", SEED)
assert ROOT is not None and commit != "unavailable", "Clone the repository first; see eval/PART2_YUYAO_BASELINE.md."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
assert torch.cuda.is_available() and torch.cuda.device_count() == 1, "Use one visible T4/P100 GPU."
for package in ("diffusers", "transformers", "peft", "accelerate", "datasets", "huggingface-hub"):
    try: print(package, metadata.version(package))
    except metadata.PackageNotFoundError: print(package, "not installed yet")

# Part 2: Yuyao Ding's partner working baseline

**Status before execution: prepared, not GPU-validated.** Team Rule 2 requires the partner to implement and run the complete pipeline at least as a working baseline. This notebook covers data preparation, one real rank-4 LoRA training run, paired base/adapter generation, KID/CLIP/LPIPS, training cost, safety checks and nearest-training-image inspection.

Pratiksha Kaushik's `p2/p2_v4.ipynb`, two 3,000-step runs and chosen rank-32 model remain the team submission. This shorter **600-step**, effective-batch-4 baseline records Yuyao's own execution. It is not a replacement final model, a new holdout, or evidence that 600 steps is a separately specified course minimum. Record actual participation and the real knowledge-transfer meeting in CONTRIBUTIONS.md; do not claim execution before running it.

Open this notebook from the cloned repository in a **fresh single Colab T4 or Kaggle T4/P100 runtime**. The first cell prints the actual commit, Python/Torch, GPU and seed. Save the executed notebook after the run. No teammate Drive directory is used.

## 1. Prepare dependencies without replacing CUDA/PyTorch

The ML API pins below match the versions printed in our earlier real T4 Part 3 run; the new Part 2 workflow still needs its own GPU run. The setup constrains existing Torch, torchvision, torchaudio, NumPy and pandas. It removes an optional incompatible torchao installation only in this dedicated cloud GPU session. It never installs Torch or changes a Windows environment.

If the cell asks for a restart because a changed package was already imported, restart once and rerun from cell 1. Hugging Face credentials, if needed, may be supplied as a Colab secret `HF_TOKEN`; they are never printed or written to artifacts.

In [ ]:
from packaging.version import Version
from src.p2_partner_baseline import gpu_check
print("Hardware check:", gpu_check())
assert Path("/content").is_dir() or Path("/kaggle").is_dir(), "Use a fresh Colab/Kaggle GPU runtime for setup."
requirements = ROOT / "config/requirements-part2-baseline.txt"
lines = [line.strip() for line in requirements.read_text().splitlines()
         if line.strip() and not line.lstrip().startswith("#")]
changed = []
for line in lines:
    name, target = line.split("==")
    try: current = metadata.version(name)
    except metadata.PackageNotFoundError: current = None
    if current != target: changed.append(name)
protected = {}
for name in ("torch", "torchvision", "torchaudio", "numpy", "pandas"):
    try: protected[name] = metadata.version(name)
    except metadata.PackageNotFoundError: pass
setup_dir = ROOT / ".runtime/part2-setup"
setup_dir.mkdir(parents=True, exist_ok=True)
constraint = setup_dir / "preserve-existing-packages.txt"
constraint.write_text("\n".join(f"{name}=={version}" for name, version in protected.items()) + "\n")
restart_needed = False
try:
    if Version(metadata.version("torchao")) < Version("0.16.0"):
        restart_needed = "torchao" in sys.modules
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
except metadata.PackageNotFoundError:
    pass
if changed:
    loaded = {"diffusers": "diffusers", "transformers": "transformers", "peft": "peft",
              "accelerate": "accelerate", "datasets": "datasets", "huggingface-hub": "huggingface_hub",
              "torchmetrics": "torchmetrics", "torch-fidelity": "torch_fidelity", "Pillow": "PIL",
              "lpips": "lpips", "ImageHash": "imagehash"}
    restart_needed |= any(loaded.get(name, name) in sys.modules for name in changed)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements),
                    "-c", str(constraint)], check=True)
assert all(metadata.version(name) == version for name, version in protected.items()), "Protected package changed."
if restart_needed:
    raise RuntimeError("Setup finished. Restart the session once, then rerun from cell 1.")
for name in ("diffusers", "transformers", "peft", "accelerate", "datasets", "torchmetrics"):
    print(name, metadata.version(name))
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        token = userdata.get("HF_TOKEN")
        if token: os.environ["HF_TOKEN"] = token
        del token
except Exception:
    pass
from diffusers import StableDiffusionPipeline
from peft import LoraConfig
from torchmetrics.image.kid import KernelInceptionDistance
import lpips, datasets, imagehash
print("Dependency imports: PASSED")

## 2. Freeze this run and its provenance

The committed 20 prompts reuse Pratiksha's previously fixed prompts, with attribution. They are fixed before this new training and remain development prompts. The in-domain captions must actually occur in this run's test split. Base, safety and CLIP model revisions and the dataset revision are resolved once and recorded.

Default artifacts are written under the ignored `.runtime/part2-yuyao/`. To retain a run on a mounted persistent volume, set `DATA266_P2_OUTPUT_DIR` to your own new run folder before this cell. To resume, set it to the same folder; config, prompt and source hashes must match. Never point it at your teammate's output directory. Colab/Kaggle temporary storage may disappear after a session expires, so download the final bundle promptly.

In [ ]:
import json
from src.p2_partner_baseline import BaselineConfig, start_run, read, fetch_trainer, sha
CFG = BaselineConfig(train_steps=600)
RUN = start_run(ROOT, output=os.environ.get("DATA266_P2_OUTPUT_DIR"), config=CFG)
trainer = fetch_trainer(RUN)
print("Verified official trainer SHA256:", sha(trainer))
print("Run directory:", RUN)
print(json.dumps(read(RUN / "manifest.json"), indent=2))

## 3. Prepare the data

We inspect image bytes and captions, remove MD5 and perceptual-hash duplicates, split 90/10 with seed 266, center-crop/resize to 512×512, and screen training images with the SD safety checker. Flagged training images are kept outside the actual training directory and never enter the trainer. The test set is not used for gradients. The printed report is computed, not copied from the teammate's counts.

Dataset limitations: Naruto screenshots and automatically generated captions can overrepresent certain characters and views; a conservative NSFW checker can remove benign anime images. No unrestricted data license is assumed. Use only for non-commercial research under the model/data terms.

In [ ]:
from src.p2_partner_baseline import prepare_data
data_report = prepare_data(RUN)
print(json.dumps(data_report, indent=2))

## 4. Train one rank-4 LoRA

The same frozen UNet attention targets and diffusion-noise objective as the teammate's workflow are implemented using the official Diffusers SD1.5 LoRA trainer. The upstream example is pinned by commit and SHA-256; attribution and source are in the runbook. It trains only the adapter, using fp16 frozen weights, fp32 trainable adapters, gradient checkpointing, micro-batch 1 and accumulation 4. The budget is **600 optimizer steps**, LR 1e-4, cosine schedule, seed 266.

This starts real training. Logs stream below and checkpoints are saved every 100 steps. If training is interrupted, retain the run folder and rerun this cell with the same RUN to resume. A completed run is reused only after verifying its adapter hash. Duration, peak memory and actual saved adapter parameters are measured in the GPU worker, not inferred from another run.

In [ ]:
from src.p2_partner_baseline import train_lora
training_stats = train_lora(ROOT, RUN)
print(json.dumps(training_stats, indent=2))

## 5. Generate a paired base/adapter comparison

Generate 20 prompts × seeds 0 and 1 for each model: 40 base images and 40 images with **your newly trained adapter**. Both use DPM-Solver++, 30 steps, CFG 7.5 and 512×512. The first pass uses a fresh base pipeline without a LoRA. Output safety checks stay enabled; flagged returned images may be blacked out, and their flags are retained. Metrics include the returned safety-filtered images, so do not directly pool these results with a pipeline that scores unfiltered images.

Each image's identity, hash and timing is saved immediately. A restart reuses validated saved images and generates only missing combinations.

In [ ]:
from src.p2_partner_baseline import generate
generations = generate(RUN)
print("Images:", len(generations))
print("Flagged:", sum(row["nsfw"] for row in generations))

## 6. Evaluate from your generated artifacts

KID uses all 20 in-domain generated images and the held-out test images, with 100 subsets; it is noisy at this size. CLIP score is 100 × the nonnegative normalized image/text cosine using CLIP ViT-B/16. LPIPS uses AlexNet on each seed pair. Costs are measured training GPU wall-hours and peak allocated VRAM. Separately compute nearest-training-image similarities for 10 adapted in-domain images; similarity is a diagnostic, not proof of memorization or its absence. No human ratings are synthesized.

In [ ]:
import pandas as pd
from src.p2_partner_baseline import evaluate
results = evaluate(RUN)
columns = ["model", "images", "kid_in_domain", "kid_std", "clip_score", "lpips",
           "nsfw_flagged", "median_seconds_per_image", "training_gpu_hours",
           "training_peak_allocated_gib", "generation_peak_allocated_gib"]
display(pd.DataFrame(results)[columns])
print("Metrics saved to:", RUN / "results.json")

## 7. Inspect side-by-side outputs and failure cases

Inspect actual outputs before making claims about domain style, objects, text, or safety. The fixed eight-prompt grid includes in-domain, OOD and compositional prompts. A second grid shows the adapted model's lowest CLIP-score prompts; this ranking is not a human pass/fail judgment. Write your own observations after viewing these images.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
rows = read(RUN / "generations.json")
lookup = {(r["model"], r["prompt_id"], r["seed"]): r for r in rows}
def show_pairs(ids):
    fig, axes = plt.subplots(len(ids), 2, figsize=(9, 4 * len(ids)), squeeze=False)
    for i, pid in enumerate(ids):
        for j, model in enumerate(("base", "yuyao_lora_r4")):
            row = lookup[(model, pid, 0)]
            axes[i, j].imshow(Image.open(RUN / row["file"]))
            axes[i, j].set_title(f"{model} | p{pid} | NSFW={row['nsfw']}")
            axes[i, j].axis("off")
        axes[i, 0].set_ylabel(lookup[("base", pid, 0)]["prompt"])
    plt.tight_layout(); plt.show()
show_pairs([0, 1, 2, 3, 10, 12, 15, 16])
worst = sorted([r for r in rows if r["model"] == "yuyao_lora_r4" and r["seed"] == 0],
               key=lambda r: r["clip_score"])[:3]
show_pairs([r["prompt_id"] for r in worst])

In [ ]:
neighbors = sorted(read(RUN / "memorization_neighbors.json"),
                   key=lambda r: r["cosine"], reverse=True)
display(pd.DataFrame(neighbors))
fig, axes = plt.subplots(3, 2, figsize=(9, 12))
for i, row in enumerate(neighbors[:3]):
    for j, field in enumerate(("generated_file", "training_file")):
        axes[i, j].imshow(Image.open(RUN / row[field]))
        axes[i, j].set_title(f"{'Generated' if j == 0 else 'Nearest training'} | cosine={row['cosine']:.3f}")
        axes[i, j].axis("off")
plt.tight_layout(); plt.show()

## 8. Yuyao's own interpretation and participation record

Complete after inspecting the results. Do not replace these items with invented observations:

- Which examples gained a recognizable domain style, and which did not?
- Give two concrete failure examples and their prompt IDs.
- Discuss whether prompt alignment/diversity changed, including noisy KID and safety-filter effects.
- Inspect the three closest generated/training pairs; do not infer “no memorization” from a threshold alone.
- Explain how the frozen VAE/text encoder, LoRA layers and diffusion-noise loss work.
- Record the real knowledge-transfer meeting date/topic/recording link with Pratiksha.

This notebook's existence does not establish that Yuyao ran it. Keep actual executed outputs and update contributions only after the real GPU run. This baseline has no new human-preference scores; the team's separate 25-pair, three-rater evaluation remains its own requirement.

## 9. Save the evidence bundle

The private ZIP includes this executed run's metadata, metrics, adapter and generated images, but excludes original training/test images and optimizer checkpoints. Keep the run directory on persistent storage if you may need to resume training. Save/download this notebook itself from Colab after execution; the source notebook file on disk does not automatically receive browser outputs.

Review the bundle before publishing. Original data stays private; adapter weights and generated image files belong on Hugging Face, not in GitHub. Nothing is uploaded automatically.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
manifest = read(RUN / "manifest.json")
assert manifest["status"] == "completed"
archive = RUN.parent / f"Part2_Yuyao_Baseline_{RUN.name}.zip"
files = [p for p in RUN.glob("*.json")] + [RUN / "training.log"]
files += list((RUN / "images").glob("*.png"))
files += list((RUN / "adapter").glob("*.safetensors"))
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in files:
        assert path.is_file(), path
        bundle.write(path, arcname=str(Path(RUN.name) / path.relative_to(RUN)))
print("Bundle:", archive)
print("Size MiB:", round(archive.stat().st_size / 1024**2, 2))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))